In [1]:
import pandas as pd
import numpy as np

# 1. We are simulating 20,000 total users (10k in Group A, 10k in Group B)
np.random.seed(42)

experiment_data = {
    'user_id': range(1, 20001),
    'device': np.random.choice(['Mobile', 'Desktop'], 20000, p=[0.7, 0.3]), # 70% use Mobile!
    'variant': ['Control'] * 10000 + ['Treatment'] * 10000,

    # 2. We simulate the purchases!
    # Control has a 7.8% conversion rate. Treatment has a 9.3% conversion rate.
    'purchased': np.concatenate([
        np.random.binomial(1, 0.078, 10000),
        np.random.binomial(1, 0.093, 10000)
    ])
}

# 3. Create the DataFrame and look at the first 5 rows
df_ab = pd.DataFrame(experiment_data)
print(df_ab.head())

# 4. Save it to a CSV for Tableau and Excel!
df_ab.to_csv('luxecart_ab_test.csv', index=False)
print("\n✅ Dataset successfully generated and saved to CSV!")

   user_id   device  variant  purchased
0        1   Mobile  Control          0
1        2  Desktop  Control          0
2        3  Desktop  Control          0
3        4   Mobile  Control          0
4        5   Mobile  Control          0

✅ Dataset successfully generated and saved to CSV!


In [2]:
from statsmodels.stats.proportion import proportions_ztest

# 1. Summarize the experiment using Lego Block #8!
summary = df_ab.groupby('variant').agg(
    total_visitors=('user_id', 'count'),
    total_buyers=('purchased', 'sum')
)

# 2. Calculate the Conversion Rate (CVR %) for each group
summary['conversion_rate_pct'] = (summary['total_buyers'] / summary['total_visitors']) * 100

print("--- EXPERIMENT PERFORMANCE TABLE ---")
print(summary)

# 3. Extract the numbers for the 2-Sample Z-Test
conversions = [summary.loc['Treatment', 'total_buyers'], summary.loc['Control', 'total_buyers']]
total_users = [summary.loc['Treatment', 'total_visitors'], summary.loc['Control', 'total_visitors']]

# 4. Run the Statistical Test (Is Treatment significantly larger than Control?)
z_stat, p_val = proportions_ztest(count=conversions, nobs=total_users, alternative='larger')

# 5. Calculate Relative Lift %
control_cvr = summary.loc['Control', 'conversion_rate_pct']
treatment_cvr = summary.loc['Treatment', 'conversion_rate_pct']
lift_pct = ((treatment_cvr - control_cvr) / control_cvr) * 100

print("\n--- STATISTICAL VERDICT ---")
print(f"Control Conversion Rate:   {control_cvr:.2f}%")
print(f"Treatment Conversion Rate: {treatment_cvr:.2f}%")
print(f"Observed Relative Lift:    +{lift_pct:.2f}%")
print(f"p-value:                   {p_val:.5f}")

if p_val < 0.05:
    print("\n✅ VERDICT: STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("🚀 RECOMMENDATION: Ship the 1-Click Checkout to 100% of users!")
else:
    print("\n🛑 VERDICT: NOT STATISTICALLY SIGNIFICANT (p >= 0.05)")
    print("❌ RECOMMENDATION: Do not ship. Risk of random chance.")

--- EXPERIMENT PERFORMANCE TABLE ---
           total_visitors  total_buyers  conversion_rate_pct
variant                                                     
Control             10000           753                 7.53
Treatment           10000           945                 9.45

--- STATISTICAL VERDICT ---
Control Conversion Rate:   7.53%
Treatment Conversion Rate: 9.45%
Observed Relative Lift:    +25.50%
p-value:                   0.00000

✅ VERDICT: STATISTICALLY SIGNIFICANT (p < 0.05)
🚀 RECOMMENDATION: Ship the 1-Click Checkout to 100% of users!
